In [0]:
PATH_RAW_DATASET = '../src/data/creditcard.csv'
PROJECT_NAME = "pork_credit_card_fraud_detection"
TBL_RAW_DATASET = f"ctl_training_dev.m7_dev.{PROJECT_NAME}_raw_dataset"
TBL_DATASET = f"ctl_training_dev.m7_dev.{PROJECT_NAME}_dataset"
EXP_NAME = f"/Workspace/Users/umaporpa@ais.co.th/cicd-lab/src/experiment/{PROJECT_NAME}"
MODEL_NAME = PROJECT_NAME
RUN_NAME = MODEL_NAME

In [0]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split

import mlflow
from mlflow.tracking import MlflowClient
from mlflow.models import infer_signature

In [0]:
df_dataset = spark.table(TBL_DATASET)
df_dataset = df_dataset.toPandas()

In [0]:
feature_names = df_dataset.iloc[:, :29].columns
target = df_dataset.iloc[:, -1:].columns


data_features = df_dataset[feature_names]
data_target = df_dataset[target]
X_train, X_test, y_train, y_test = train_test_split(data_features, data_target, 
                                                    train_size = 0.70, test_size = 0.30)

In [0]:
rf_classifier = RandomForestClassifier(
    n_estimators=100,       
    max_depth=10,           
    min_samples_split=0.1,    
    n_jobs=-1               
)

rf_classifier.fit(X_train, y_train)

In [0]:
lgr_model = LogisticRegression()
lgr_model.fit(X_train, y_train)

In [0]:
rf_train_pred = rf_classifier.predict_proba(X_train)
rf_test_pred = rf_classifier.predict_proba(X_test)

lgr_train_pred = lgr_model.predict_proba(X_train)
lgr_test_pred = lgr_model.predict_proba(X_test)

# Register Model

In [0]:
mlflow.set_experiment(EXP_NAME)

In [0]:
with mlflow.start_run(run_name="Fraud_detect_Random_Forest") as rf_run:

    rf_train_roc_auc = roc_auc_score(y_train, rf_train_pred[:,-1])
    rf_test_roc_auc = roc_auc_score(y_test,rf_test_pred[:,-1])

    rf_train_pr_auc = average_precision_score(y_train, rf_train_pred[:,-1])
    rf_train_pr_auc = average_precision_score(y_test, rf_test_pred[:,-1])
    
    signature = infer_signature(X_train, rf_train_pred)
    mlflow.log_metric("Train_ROC_AUC", rf_train_roc_auc)
    mlflow.log_metric("Test_ROC_AUC", rf_test_roc_auc)
    mlflow.log_metric("Train_PR_AUC", rf_train_pr_auc)
    mlflow.log_metric("Test_PR_AUC", rf_train_pr_auc)
    
    mlflow.sklearn.log_model(
        sk_model=rf_classifier,
        artifact_path="rf_model_artifact",
        #registered_model_name="Random_Forest_Model",
        input_example=X_train[:5],
        signature=signature
    )

In [0]:
with mlflow.start_run(run_name="Fraud_detect_Log_Reg") as lgr_run:

    lgr_train_roc_auc = roc_auc_score(y_train, lgr_train_pred[:,-1])
    lgr_test_roc_auc = roc_auc_score(y_test,lgr_test_pred[:,-1])

    lgr_train_pr_auc = average_precision_score(y_train, lgr_train_pred[:,-1])
    lgr_train_pr_auc = average_precision_score(y_test, lgr_test_pred[:,-1])
    
    signature = infer_signature(X_train, lgr_train_pred)
    mlflow.log_metric("Train_ROC_AUC", lgr_train_roc_auc)
    mlflow.log_metric("Test_ROC_AUC", lgr_test_roc_auc)
    mlflow.log_metric("Train_PR_AUC", lgr_train_pr_auc)
    mlflow.log_metric("Test_PR_AUC", lgr_train_pr_auc)
    
    mlflow.sklearn.log_model(
        sk_model=rf_classifier,
        artifact_path="lgr_model_artifact",
        registered_model_name="Log_reg_Model",
        input_example=X_train[:5]
    )

In [0]:
client = MlflowClient()

experiment = client.get_experiment_by_name(EXP_NAME)
runs = client.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.Test_ROC_AUC"], 
    max_results=1,
)

best_run = runs[0]
best_run_id = best_run.info.run_id
model_uri = f"runs:/{best_run_id}/model"

model_name = "Advanced_State_of_the_Art_Rule_Base"
model_version = mlflow.register_model(model_uri=model_uri, name=model_name)